# Session 2 - Data Quality, the Split-First Rule & Visual Reasoning

**Block 1: Data Science Fundamentals** · 4 hours

---

## Learning objectives

By the end of this session you will be able to:

1. Split data **before** inspecting the target, and explain what the rule protects. `[CLO3]`
2. Choose a splitting strategy that respects group structure, and justify it from
   evidence in the data rather than from a rule of thumb. `[CLO4]`
3. Diagnose and document six categories of data-quality defect. `[CLO2]`
4. Classify missingness by **mechanism**, and *measure* the bias that dropping rows
   introduces. `[CLO2]`
5. Select a visualization from a stated question, and say what a given chart
   cannot show. `[CLO10]`

## Prerequisites

Session 1, and your Dataset Fact Sheet (M0) open beside you. You need three facts
from it: one row is one listing; there are 4,595 hosts for 15,293 listings; and
twelve columns are empty.

## Why does this matter?

Two reasons, and the second one is the one people underestimate.

**First:** every number you report for the next ten weeks depends on a decision you
make in the first forty minutes of today. Get the split wrong and nothing
downstream can be trusted, no matter how careful you are afterwards.

**Second:** cleaning is not tidying. Every cleaning decision is a **modelling
decision in disguise**, and several of them change your conclusions. By the end of
today you will have measured that, not been told it.

## §1 - Retrieval practice

Answer from memory. No scrolling, no running code. Five minutes.

1. One row of this dataset is one ______.
2. There are 15,293 listings and ______ hosts. The largest host owns ______ listings.
3. At which workflow stage does the train/test split belong?
4. How many of the 90 columns are entirely empty?
5. Name one question this data cannot answer, and why.

In [ ]:
import sys
from pathlib import Path

# Same root-finding walk as Session 1: make src/ importable wherever we launched.
ROOT = Path.cwd()
while not (ROOT / "src").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# split_by_host is the grouped split we are about to justify from first principles.
from src.data import PATHS, SEED, load_raw, set_seed, split_by_host

# One consistent visual style for every chart in the session, set once.
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)
# Seed before anything random happens, including the split below.
set_seed(SEED)

df = load_raw()
print(f"loaded {df.shape[0]:,} rows x {df.shape[1]} columns")

## §2 - The split, first

### The rule

> **Split the data before you look at the target. Set the test half aside. Do not
> open it again until the last session of the course.**

Everything you do afterwards - cleaning, imputation, feature construction, model
choice, hyperparameters - must be decided using the training half only.

### Why

The purpose of a test set is to estimate how your model behaves on data it has
never influenced. The moment a decision of yours is informed by the test half,
that half stops being an estimate of the future and becomes part of your training
process. It will flatter you, and you will not know by how much.

The subtle part: this includes decisions that don't *feel* like training. Choosing
a median to impute with. Deciding a price of €10,542 is an error. Noticing a
skewed distribution and taking a log. Each of those uses information, and if it
uses information from the test half, your final number is a lie you told yourself.

### But *how* do we split?

This is where Session 1's finding earns its keep. Before you split, look at the
structure of the rows.

In [ ]:
# The structure that decides how we are allowed to split. Counting rows per host is
# the whole argument: if one host owns hundreds of listings, those rows are not
# independent draws, and a random split will scatter one host across both halves.
hosts = df["host_id"].value_counts()
print(f"listings                    : {len(df):,}")
print(f"hosts                       : {hosts.size:,}")
# hosts > 1 selects the multi-listing hosts; .sum() over their counts converts that
# back into a number of rows, which is the share of the data at risk.
print(f"listings from multi-listing hosts: {hosts[hosts > 1].sum():,} "
      f"({hosts[hosts > 1].sum() / len(df) * 100:.1f}%)")
print(f"largest single host         : {hosts.max()} listings")

### Predict before you run

Suppose we ignore all of that and split at random - every row gets an independent
coin flip, 80% train, 20% test.

**Write your prediction:** of the ~3,059 listings that land in the test set, how
many will belong to a host who *also* has listings in the training set?

In [ ]:
# TODO: Write your prediction as a number or a percentage, then run the next cell.
#
# My prediction: ____ of ~3,059 test listings will share a host with the train set.

In [ ]:
from sklearn.model_selection import train_test_split

# The default split everyone reaches for, done correctly as far as the API goes:
# 20% held out, seeded, reproducible. The problem is not the code, it is the
# assumption underneath it that rows are interchangeable.
naive_train, naive_test = train_test_split(df, test_size=0.2, random_state=SEED)
# Set intersection: which host ids ended up on both sides of the wall.
shared_hosts = set(naive_train["host_id"]) & set(naive_test["host_id"])
# isin() turns that set into a row-level mask over the test half.
contaminated = naive_test["host_id"].isin(shared_hosts)

print("NAIVE RANDOM SPLIT")
print(f"  hosts appearing in BOTH halves : {len(shared_hosts):,}")
# Read this percentage out loud before revealing it. It is the point of the section:
# these test rows have a sibling in training, so the test set is measuring memory.
print(f"  test listings whose host is also in train: "
      f"{contaminated.sum():,} of {len(naive_test):,} ({contaminated.mean() * 100:.1f}%)")

**77.9%.**

Four out of five listings in that test set belong to an operator the model has
already studied. So what would such a test set measure? Not "can this model price
an unfamiliar listing" - it measures "can this model recall what this particular
management company charges." Those are different questions, and only one of them
is the client's.

The fix is to split by **host**, not by row: whole hosts go to one side or the
other.

In [ ]:
# TODO: Split by host so that no host_id appears in both halves.
#
# Use sklearn.model_selection.GroupShuffleSplit with:
#     n_splits=1, test_size=0.2, random_state=SEED
# and groups=df["host_id"].
#
# Then print: train rows, test rows, train hosts, test hosts, and the number of
# hosts appearing in both (which must be 0).

train = None   # replace
test = None    # replace

Notice something: we asked for 20% and got **17.4%**. That is not a bug. Whole
hosts move together, and one of them carries 588 listings, so the fraction cannot
land exactly where you asked. Report what you got, not what you requested.

### Sealing the test set

Now the ritual. We write the test half to disk and we do not read that file again
until Session 12.

In [ ]:
# Sealing the test set. Writing it to disk and deleting the variable is the whole
# ceremony: from here the file exists, and the rule is that nobody opens it until
# Session 12.
PATHS.sealed.mkdir(parents=True, exist_ok=True)
sealed_path = PATHS.sealed / "test.parquet"
# Parquet rather than CSV: it preserves dtypes, so the sealed set comes back in
# Session 12 exactly as it went in, with no re-parsing to get wrong.
test.to_parquet(sealed_path, index=False)

print(f"sealed {len(test):,} rows -> {sealed_path.relative_to(PATHS.root)}")
print(f"file size: {sealed_path.stat().st_size / 1e6:.1f} MB")
print("\nFrom this point on, `train` is your entire world.")

# Free the variable so an accidental reference fails loudly rather than quietly.
del test

That `del test` is not theatre. If you keep the object in memory you *will*
eventually type `test` into a cell at 1am and it will work. Make the accident
raise a `NameError`.

## §3 - Six categories of data-quality defect

From here on: **training data only.**

"Clean the data" is not an instruction, it is a category. Here are six specific
things to look for, each with a different remedy and a different risk.

| # | Defect | Question to ask |
|---|---|---|
| 1 | Wrong type | Can I do arithmetic on this? Should I be able to? |
| 2 | Structured value in one cell | Is this one fact or a collection? |
| 3 | Impossible values | What does the domain say is possible? |
| 4 | Outliers | Is this an error, or a real extreme? |
| 5 | Missing values | *Why* is it missing? |
| 6 | Deceptive duplicates | Are these the same thing, or two things that look alike? |

### Defect 1 - wrong type

In [ ]:
# Why price cannot be used as it stands. The dtype says "object", which for pandas
# means text, and the examples show why: a currency symbol and thousands separators.
print("price, as stored:")
print(f"  dtype   : {train['price'].dtype}")
print(f"  examples: {train['price'].dropna().head(4).tolist()}")

# Deliberately triggering the error rather than describing it. Let the class read
# the actual TypeError, because that is the message they will meet on their own.
try:
    train["price"].mean()
except TypeError as exc:
    print(f"\n  train['price'].mean() -> TypeError: {exc}")

In [ ]:
# Parsing price, one step at a time, and each step is a decision.
price = (
    train["price"]
    # astype(str) so the regex applies uniformly even where pandas stored NaN.
    .astype(str)
    # Strip everything that is not a digit or a decimal point: the currency symbol
    # and the thousands separators go, the number survives.
    .str.replace(r"[^0-9.]", "", regex=True)
    # A row that was empty to begin with is now the empty string, which would crash
    # the conversion. Send it back to NaN, where it belongs.
    .replace("", np.nan)
    .astype(float)
)
# assign() returns a new frame rather than mutating train in place, which keeps the
# original column available for comparison.
train = train.assign(price_num=price)

print(f"parsed {price.notna().sum():,} prices; {price.isna().mean() * 100:.1f}% missing")
# The percentiles matter more than the mean here. The 99th against the max is the
# first sign of how long the right tail is.
print(price.describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.99]).round(2).to_string())

### Defects 3 and 4 - impossible values, and outliers

These are different problems and conflating them is a classic error. An
*impossible* value violates the domain (a negative price, 0 guests). An *outlier*
is possible but extreme - and extremes are often the most interesting rows in the
dataset.

In [ ]:
# Implausible values, counted rather than eyeballed. Each line below is a different
# kind of defect, and none of them is a missing value.
print(f"prices over  €1,000 : {(price > 1000).sum():,}")
print(f"prices over  €5,000 : {(price > 5000).sum():,}")
print(f"maximum price       : €{price.max():,.2f}")
# Zero is not missing, it is a claim. A free flat is a data defect, not a bargain.
print(f"prices at zero      : {(price == 0).sum()}")
# A listing that sleeps nobody is impossible, so this is a recording artifact.
print(f"accommodates == 0   : {(train['accommodates'] == 0).sum()}")
# A minimum stay beyond a year is how a host hides a listing without delisting it.
# The value is real, but it does not mean what the column name suggests.
print(f"minimum_nights > 365: {(train['minimum_nights'] > 365).sum()} "
      f"(max {train['minimum_nights'].max():.0f})")

# nlargest is the fastest way to look at the tail. Look at the other columns too:
# the question is whether these listings are errors or simply unusual properties.
train.nlargest(5, "price_num")[
    ["price_num", "room_type", "accommodates", "bedrooms", "neighbourhood_cleansed"]
]

Look at those five rows before deciding anything. Some are plausible luxury
properties; at least one is hard to defend. You have to make a call, and the call
has to go in your decision log.

There is no correct answer here. There is only a documented answer and an
undocumented one.

### Defect 6 - deceptive duplicates

In [ ]:
# Duplicates, and why the obvious test finds nothing useful here.
# duplicated() compares whole rows, and whole rows differ by id, so this is near zero.
print(f"exact duplicate rows: {train.duplicated().sum()}")

# The interesting test uses a business key: same coordinates, same room type, same
# capacity. Two rows matching on all four are describing the same physical space.
key = ["latitude", "longitude", "room_type", "accommodates"]
# keep=False marks every member of a duplicated group, not just the repeats, which
# is what we want when we are trying to see the groups themselves.
dup_mask = train.duplicated(subset=key, keep=False)
print(f"rows sharing (lat, lon, room_type, accommodates): {dup_mask.sum():,}")

# The decisive question: are these one host listing a unit several times, or
# different hosts colliding by chance? Counting distinct hosts per group answers it.
groups = train[dup_mask].groupby(key)["host_id"].nunique()
print(f"  such groups              : {len(groups):,}")
# A group with one host is a multi-unit operator, not a data error. Deleting these
# rows would quietly delete the professional operators, which is exactly the
# population the regulator cares about.
print(f"  groups with a single host: {(groups == 1).sum():,} "
      f"({(groups == 1).mean() * 100:.0f}%)")

So they are overwhelmingly **one host with several identical units at one
address** - an aparthotel or a managed building. These are real, distinct,
separately bookable listings.

`train.drop_duplicates(subset=key)` would delete them. That is not cleaning; it is
deleting the professional operators from a dataset whose whole point is to study
professional operators.

> **The habit to build:** before dropping anything, look at what you are about to
> drop and ask what it has in common.

## §4 - Guided exercise: build a cleaning function with a decision log

Write one function that performs your cleaning, and one table that justifies it.

The log has four columns, and the fourth is the one that distinguishes a
professional from an enthusiast:

| what I did | why | what I'd have lost otherwise | what this could bias |
|---|---|---|---|

In [ ]:
# TODO: Complete the cleaning function.
#
# Requirements:
#   - parse price into a numeric column
#   - drop columns that are entirely empty, and constant columns
#   - handle bathrooms_text -> a numeric bathroom count (look at the values first!)
#   - decide what to do about extreme prices, and DO NOT silently drop rows
#   - return the cleaned frame; do not mutate the input
#
# Rules:
#   - every decision gets a line in your log
#   - no step may use information from outside `df`


def clean(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    # 1. price -> numeric
    # TODO

    # 2. drop empty and constant columns
    # TODO

    # 3. bathrooms_text -> numeric
    # TODO

    # 4. extreme prices: flag, don't silently drop
    # TODO

    return out

## §5 - Missingness lab: two mechanisms, measurably different

"Fill missing values with the mean" is the most widely repeated bad advice in
applied statistics. Whether it is defensible depends entirely on **why** the value
is missing, and that is a question about the world, not about the dataframe.

We have two columns with substantial missingness. They look similar in a summary
table and they are nothing alike.

In [ ]:
# Two columns with missing values, and the rates alone tell you nothing about what
# to do. The mechanism behind the gap is what decides the treatment.
for col in ["review_scores_rating", "price"]:
    print(f"{col:24s} {df[col].isna().mean() * 100:5.1f}% missing")

### Case 1 - `review_scores_rating`

Investigate *why* it is missing before deciding what to do.

In [ ]:
# Diagnosing the mechanism: compare the rows that are missing against the rows that
# are not, on a column that might explain the gap.
miss = train["review_scores_rating"].isna()
print("mean number_of_reviews:")
# .loc[mask, col] selects the rows where the mask is True and that one column.
print(f"  where rating is missing: {train.loc[miss, 'number_of_reviews'].mean():.2f}")
print(f"  where rating is present: {train.loc[~miss, 'number_of_reviews'].mean():.2f}")
print()
# The decisive line. A rating is missing because there were no reviews to average,
# so the value is not unknown, it is undefined. Imputing the mean rating here would
# invent a reputation for a listing that has never been rated.
print(f"share with ZERO reviews, among missing: "
      f"{(train.loc[miss, 'number_of_reviews'] == 0).mean() * 100:.1f}%")
print(f"share with ZERO reviews, among present: "
      f"{(train.loc[~miss, 'number_of_reviews'] == 0).mean() * 100:.1f}%")

**100% and 0%.** The missingness is not noise - it is a *definition*. A listing
has no rating precisely when nobody has reviewed it.

So imputing the mean rating (about 4.8) asserts that unreviewed listings are
averagely good. They are not averagely good; they are **unrated**, which is a
different state and one a guest can see. The information here is the missingness
itself.

The defensible handling: an explicit `has_reviews` indicator, and leave the score
missing for a model that can handle it natively.

### Case 2 - `price` - now measure the bias

### Predict before you run

12.7% of listings have no price. The obvious move is to drop those rows - you
cannot model a target you do not have.

**Write your prediction:** are the dropped rows a random sample of the data, or
systematically different? If different, in what way?

In [ ]:
# TODO: Prediction, then investigate.
#
# My prediction:
#
# Then compare the price-missing rows against the price-present rows on at least
# three columns of your choosing. Quantify each difference.

Read the availability row: listings with no price are available **39.6** days a
year on average; those with a price, **237.5**. And 60.9% of them are private
rooms, against 27.9% in the dataset overall.

These are not missing at random. They are overwhelmingly **inactive or
barely-active private rooms** - a listing you cannot get a price quote for because
it is effectively not on the market.

So dropping them is *defensible*, but it silently redefines your problem. You are
no longer modelling "the price of a Barcelona listing"; you are modelling "the
price of an actively marketed Barcelona listing", which skews toward entire homes
with high availability.

That is fine. It may even be what Client B wants. But it has to be **written
down**, because it changes who your model applies to - and someone will eventually
ask.

## §6 - Visual reasoning

The question is never "how do I draw a histogram". It is **"what do I want to
know, and which picture would answer it?"**

Work up the ladder. Each rung answers a different kind of question.

| Rung | Question type | Typical chart |
|---|---|---|
| Univariate | How is one thing distributed? | histogram, boxplot, ECDF |
| Bivariate | Do two things move together? | scatter, boxplot-by-group |
| Multivariate | Does the relationship depend on a third thing? | facets, hue, heatmap |

And for every chart you make, answer two questions: *what does this show?* and
*what does this specifically fail to show?*

### Univariate: the shape of the target

In [ ]:
# Why a transformation, shown rather than asserted. between() scopes to a plausible
# range so the chart is about the shape of the distribution, not about one outlier.
valid = train.loc[train["price_num"].between(10, 2000), "price_num"]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
# Left: the raw prices. skew() is printed in the title so the shape has a number
# attached to it rather than only an impression.
axes[0].hist(valid, bins=60, color="steelblue")
axes[0].set(title=f"price (skew = {valid.skew():.2f})", xlabel="€ per night",
            ylabel="listings")
# Right: the same values through log1p, which is log(1 + x) and so is defined at
# zero. On a multiplicative quantity such as price this turns a long right tail
# into something close to symmetric, which is what linear models expect.
axes[1].hist(np.log1p(valid), bins=60, color="darkorange")
axes[1].set(title=f"log1p(price) (skew = {np.log1p(valid).skew():.2f})",
            xlabel="log(1 + €)")
# Same data in both panels. Only the scale changed, and the skew numbers show how
# much that alone bought us.
fig.suptitle("The same data, twice", y=1.02)
plt.tight_layout()
plt.show()

Skew **9.04** becomes **0.03**. The right-hand shape is roughly symmetric, which
matters because several models we meet later assume errors that behave that way.

What the left chart *cannot* tell you: whether the long tail is luxury property or
data entry error. A histogram shows you that extremes exist; only looking at the
rows tells you what they are.

### Bivariate and multivariate: your turn

In [ ]:
# TODO: Produce three charts. For each one, write in a comment:
#        (a) the question it answers
#        (b) one thing it cannot tell you
#
# 1. A bivariate chart comparing log price across the four room types.
# 2. A multivariate chart: price vs accommodates, split by room type.
# 3. A chart of your own choosing that answers a question you actually have.

### Correlation is not causation, concretely

It is easy to nod at this and then forget it two cells later, so here it is in
your own data.

In [ ]:
# Correlation as a closing caution. corr() is Pearson by default, so it measures
# *linear* association only: a strong curved relationship can show near zero here.
sub = plot_df[["log_price", "number_of_reviews", "accommodates", "availability_365"]]
print("Correlation with log price:")
# Pull the target's column out of the square matrix and drop its self-correlation
# of 1.0, leaving one readable number per feature.
print(sub.corr()["log_price"].drop("log_price").round(3).to_string())

Suppose `number_of_reviews` correlates negatively with price. A tempting reading:
"getting more reviews drives your price down." A moment's thought gives at least
three rival explanations:

1. **Reverse direction** - cheap listings get booked more, so they accumulate more
   reviews. The causation runs the other way.
2. **Confounding** - private rooms are cheap *and* heavily booked. Room type drives
   both.
3. **Selection** - we dropped the listings with no price, which were mostly
   inactive private rooms. We changed the population before measuring.

Nothing in the correlation coefficient distinguishes these. Only a claim about the
world does, and this dataset - an observational scrape - cannot settle it.

## §7 - Common mistakes

| Mistake | Why it is tempting | What to do instead |
|---|---|---|
| Cleaning before splitting | Cleaning feels like a neutral prerequisite | Split at stage 3. Always. |
| Random split on grouped data | It is the default argument | Check for a grouping column first. Here, 77.9% contamination. |
| `drop_duplicates()` as hygiene | Duplicates sound obviously bad | Look at what you are deleting. 475 of 509 groups are one legitimate operator. |
| `fillna(mean)` | It always runs | Ask *why* it is missing. Sometimes the mean asserts something false. |
| Dropping rows and moving on | You cannot model a missing target | You may drop them; you must state how the population changed. |
| Charting before questioning | Plots are satisfying to make | Write the question in a comment first. |
| Reporting the requested split size | You asked for 20% | You got 17.4%. Report what happened. |

## §8 - Reflection

1. You dropped 12.7% of rows because they had no price. Write the one-sentence
   caveat that belongs in your final report because of that decision.
2. Your cleaning function flags implausible prices instead of removing them. What
   does that buy you in Session 6 that removal would have cost?
3. You have now sealed a test set. Name a specific way you could accidentally
   contaminate it in the next three weeks without noticing.

## §9 - Knowledge check

1. Why must the test set be split off before cleaning rather than after?
2. A random 80/20 split put 77.9% of test listings in the same host as a training
   listing. Explain, in one sentence, what such a test set actually measures.
3. `review_scores_rating` is missing for exactly the listings with zero reviews.
   Name the mechanism and give the defensible handling.
4. You requested a 20% test set and got 17.4%. Why, and which figure do you report?
5. Give one rival explanation, other than causation, for a negative correlation
   between price and review count.

## Summary

- **Split first.** Stage 3, before cleaning, before EDA. The test half is sealed
  until Session 12.
- **Split by the right unit.** A random split contaminated 77.9% of the test set
  via shared hosts. Grouping by `host_id` fixes it, and costs you an exact 20%.
- Cleaning is six distinct problems, not one. Wrong types, structured values,
  impossible values, outliers, missing values, deceptive duplicates.
- **Flag, do not drop.** Reversible cleaning leaves the decision to the modelling
  step where it can be tested.
- Missingness has a **mechanism**. `review_scores_rating` is missing by definition;
  `price` is missing for inactive private rooms. Neither is random and neither
  should get the mean.
- Dropping the price-missing rows was defensible and **changed the population your
  model describes**. Both halves of that sentence matter.
- Choose the chart from the question, and always state what the chart cannot show.

## Key takeaways

1. Look at what you are about to delete before you delete it.
2. "Defensible" and "harmless" are different words.
3. A correlation has at least three explanations before it has one.

## Further exploration

**Essential**
- Little & Rubin, *Statistical Analysis with Missing Data*, ch. 1 - the MCAR/MAR/MNAR
  distinction at the source.
- scikit-learn user guide, *Imputation of missing values*:
  https://scikit-learn.org/stable/modules/impute.html

**Recommended**
- Wickham (2014), *Tidy Data*, Journal of Statistical Software.
  https://vita.had.co.nz/papers/tidy-data.pdf
- Kapoor & Narayanan (2023), *Leakage and the reproducibility crisis in
  machine-learning-based science*, Patterns. https://arxiv.org/abs/2207.07048 -
  read the abstract now; we return to it next session.

**Advanced**
- Van Buuren, *Flexible Imputation of Missing Data*, 2nd ed., ch. 2.
  https://stefvanbuuren.name/fimd/ - free online.

---

**Next session:** you will engineer features, and you will build a model that
performs suspiciously well. Bring your cleaning function; you will need it.